# 3단계: 전이학습 (Transfer Learning)

ImageNet(1,400만 장)으로 사전학습된 **ResNet18**을 가져와 **CIFAR-10**(10개 클래스 컬러 이미지) 분류에 재활용합니다.

**실습 흐름**

1. **특징 추출**: 사전학습 층은 전부 동결하고 마지막 분류층만 학습
2. **파인튜닝**: 뒤쪽 층(layer4)까지 풀어서 낮은 학습률로 추가 학습 → 정확도 상승 확인

2단계와 비교해보세요. 학습 코드(4줄 패턴)는 완전히 같고, **모델을 어디서 가져와 어떻게 수정하는지**만 다릅니다.

> 첫 실행 시 CIFAR-10 데이터셋 약 170MB + ResNet18 가중치 약 45MB를 자동 다운로드합니다.
> 전체 실행 시간: RTX 2060 기준 약 15~25분 (2단계보다 오래 걸리는 것이 정상입니다)

## 0. 설정

In [1]:
import time
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 장치: {device}")

BATCH_SIZE = 64    # 224x224 이미지라 2단계보다 작게 (VRAM 6GB 기준)


사용 장치: cuda


## 1. 데이터 준비 — CIFAR-10

CIFAR-10: 32×32 컬러 이미지 10개 클래스(비행기, 자동차, 새, 고양이, 사슴, 개, 개구리, 말, 배, 트럭). 학습 50,000장 / 테스트 10,000장.

**전이학습의 핵심 규칙**: 입력을 **사전학습 때와 같은 형식**으로 맞춰야 합니다.
ResNet18은 ImageNet의 224×224 이미지, 특정 평균/표준편차로 학습됐으므로 CIFAR-10도 똑같이 변환합니다.

In [3]:
# ImageNet 사전학습 조건에 맞춘 변환
transform = transforms.Compose([
    transforms.Resize(224),                      # 32x32 -> 224x224 (ResNet 입력 크기)
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],   # ImageNet 평균
                         std=[0.229, 0.224, 0.225]),   # ImageNet 표준편차
])

train_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR10(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

classes = ["비행기", "자동차", "새", "고양이", "사슴", "개", "개구리", "말", "배", "트럭"]
print(f"학습: {len(train_dataset)}장, 테스트: {len(test_dataset)}장")

KeyboardInterrupt: 

### 데이터 확인

In [ ]:
import matplotlib.pyplot as plt

# 시각화용: 정규화 없이 원본 크기로 다시 로딩
raw_dataset = datasets.CIFAR10(root="./data", train=True, download=False,
                               transform=transforms.ToTensor())

fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for i, ax in enumerate(axes):
    image, label = raw_dataset[i]
    ax.imshow(image.permute(1, 2, 0))   # (채널,H,W) -> (H,W,채널)
    ax.set_title(classes[label])
    ax.axis("off")
plt.show()

## 2. 사전학습 모델 불러오기 + 수정

전이학습의 핵심 3단계:

1. **사전학습 가중치와 함께 모델 로드** (`weights=...`)
2. **기존 층 동결** — `requires_grad = False` 로 역전파에서 제외
3. **마지막 분류층 교체** — ImageNet 1,000개 클래스 → 우리 문제 10개 클래스

In [ ]:
# 1) ImageNet 사전학습 가중치로 ResNet18 로드
model = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)

# 2) 모든 파라미터 동결 (학습에서 제외)
for param in model.parameters():
    param.requires_grad = False

# 3) 마지막 분류층(fc)을 새 것으로 교체
#    새로 만든 층은 requires_grad=True가 기본이므로 이 층만 학습된다
print(f"교체 전 fc: {model.fc}")
model.fc = nn.Linear(model.fc.in_features, 10)   # 1000 클래스 -> 10 클래스
print(f"교체 후 fc: {model.fc}")

model = model.to(device)

# 학습되는 파라미터 수 확인
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"전체 파라미터: {total:,}개 중 학습 대상: {trainable:,}개 ({100*trainable/total:.2f}%)")

전체의 **0.05%만 학습**한다는 점에 주목하세요. 나머지 99.95%는 ImageNet에서 배운 지식을 그대로 사용합니다.

## 3. 학습/평가 함수

2단계와 완전히 같은 구조입니다. 재사용할 수 있도록 model과 optimizer를 인자로 받게만 바꿨습니다.

In [ ]:
criterion = nn.CrossEntropyLoss()

def train_one_epoch(model, optimizer, epoch):
    model.train()
    total_loss = 0
    for batch_idx, (images, labels) in enumerate(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()              # 1) 기울기 초기화
        outputs = model(images)            #    순전파
        loss = criterion(outputs, labels)  # 2) 손실 계산
        loss.backward()                    # 3) 역전파
        optimizer.step()                   # 4) 파라미터 업데이트

        total_loss += loss.item()
        if (batch_idx + 1) % 200 == 0:
            print(f"  Epoch {epoch} [{(batch_idx + 1) * BATCH_SIZE:>6}/{len(train_dataset)}] "
                  f"loss: {loss.item():.4f}")
    print(f"  -> Epoch {epoch} 평균 loss: {total_loss / len(train_loader):.4f}")


def evaluate(model):
    model.eval()
    correct = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            predicted = model(images).argmax(dim=1)
            correct += (predicted == labels).sum().item()
    acc = 100 * correct / len(test_dataset)
    print(f"  -> 테스트 정확도: {acc:.2f}%")
    return acc

## 4. 1차 학습: 특징 추출 (분류층만 학습)

동결된 층은 기울기 계산이 없어 1 에폭만으로도 꽤 높은 정확도가 나옵니다.
(RTX 2060 기준 에폭당 5~8분)

In [ ]:
# 학습 대상 파라미터만 옵티마이저에 전달
optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=0.001)

start = time.perf_counter()
train_one_epoch(model, optimizer, epoch=1)
acc_frozen = evaluate(model)
print(f"소요 시간: {time.perf_counter() - start:.0f}초")

**생각해볼 점**: 2단계 MNIST는 3 에폭으로 ~99%였지만, CIFAR-10은 훨씬 어려운 문제(컬러, 복잡한 물체)입니다.
그런데도 **단 1 에폭, 파라미터 0.05% 학습**만으로 80% 이상이 나옵니다. 밑바닥부터 학습하면 같은 정확도에 수십 에폭이 필요합니다. 이것이 전이학습의 힘입니다.

## 5. 2차 학습: 파인튜닝 (layer4까지 해제)

이제 ResNet의 마지막 블록(layer4)도 학습에 참여시킵니다.

**핵심**: 사전학습된 지식을 망가뜨리지 않도록 **학습률을 10분의 1로** 낮춥니다.

In [ ]:
# layer4 동결 해제
for param in model.layer4.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"학습 대상 파라미터: {trainable:,}개로 증가")

# 낮은 학습률로 새 옵티마이저 구성
optimizer = optim.Adam([p for p in model.parameters() if p.requires_grad], lr=0.0001)

start = time.perf_counter()
train_one_epoch(model, optimizer, epoch=2)
acc_finetuned = evaluate(model)
print(f"소요 시간: {time.perf_counter() - start:.0f}초")

print(f"\n특징 추출만: {acc_frozen:.2f}%  ->  파인튜닝 후: {acc_finetuned:.2f}%")

## 6. 예측 결과 확인 + 모델 저장

In [ ]:
fig, axes = plt.subplots(1, 8, figsize=(14, 2.2))
model.eval()
with torch.no_grad():
    for i, ax in enumerate(axes):
        image, label = test_dataset[i]          # 변환된 224x224 (모델 입력용)
        raw_img, _ = datasets.CIFAR10(root="./data", train=False, download=False,
                                      transform=transforms.ToTensor())[i]
        pred = model(image.unsqueeze(0).to(device)).argmax(dim=1).item()
        ax.imshow(raw_img.permute(1, 2, 0))
        color = "black" if pred == label else "red"
        ax.set_title(f"정답:{classes[label]}\n예측:{classes[pred]}", color=color, fontsize=9)
        ax.axis("off")
plt.tight_layout()
plt.show()

torch.save(model.state_dict(), "resnet18_cifar10.pt")
print("모델 저장 완료: resnet18_cifar10.pt")

---
**3단계 완료!**

**배운 것 정리**

- 사전학습 모델 로드 → 동결 → 분류층 교체가 전이학습의 기본 패턴
- 특징 추출(빠름, 안전) vs 파인튜닝(느림, 더 정확) 트레이드오프
- 파인튜닝 시 낮은 학습률이 필수인 이유

**도전 과제**

1. layer3까지 동결 해제하면 정확도가 더 오를까요? (VRAM 사용량도 관찰해보세요)
2. `models.resnet34` 등 더 큰 모델로 바꿔보세요
3. 학습률을 0.001로 파인튜닝하면 어떻게 되는지 실험해보세요 (사전학습 지식 파괴 체험)

다음 4단계에서는 같은 원리를 텍스트(Hugging Face 모델 파인튜닝)에 적용합니다.